<a href="https://colab.research.google.com/github/JxCorona/CIS3120/blob/main/CIS3120_Corona_Jovanny_LibraryDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Housekeeping (Step 1)



In [1]:
import sqlite3
import csv
import urllib.request
import os

# GitHub raw URLs for the three CSV files
# Replace <username> and <repo> with the actual values provided by your instructor

BASE_URL = "https://raw.githubusercontent.com/ProfessorPatrickSlatraigh/CIS3120-BMWB/refs/heads/main/"

BOOK_URL = BASE_URL + "Book.csv"
MEMBER_URL = BASE_URL + "Member.csv"
LOAN_URL = BASE_URL + "Loan.csv"

# Local paths in the Colab /content directory
BOOK_PATH = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH = "/content/Loan.csv"

DB_PATH = "/content/library.db"


Download CSV File (Step 2)

In [3]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:

    urllib.request.urlretrieve(url, path)

    print(f"Downloaded: {path}")

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


Create Database and Tables (Step 3)

In [4]:
from genericpath import exists
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

conn.execute('PRAGMA foreign_keys = ON;')
# CREATE TABLE statements go here
# ...
#BOOK TABLE | BOOK TABLE
try:
  book = """
  CREATE table IF NOT exists book (
    callNo  TEXT    NOT NULL,
    title   TEXT    NOT NULL,
    author  TEXT    NOT NULL,
    PRIMARY KEY (callNo)
    ); """

  cursor.execute(book)
#except Exception as e:
  #print(f"Error: {e}")


#MEMBER TABLE | MEMBER TABLE
  member =  """
  CREATE TABLE IF NOT EXISTS member (
    id          INTEGER NOT NULL,
    firstname   TEXT    NOT NULL,
    lastName    TEXT    NOT NULL,
    PRIMARY KEY (id)
    );"""
  cursor.execute(member)
#except Exception as e:
  #print(f"Error {e}")

#LOAN TABLE | LOAN TABLE
#try:
  loan =  """
  CREATE TABLE IF NOT EXISTS loan (
    callNo        TEXT    NOT NULL,
    id            INTEGER NOT NULL,
    dateBorrowed  TEXT    NOT NULL,
    dateReturned  TEXT,
    dateDue       TEXT    NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id)     REFERENCES Member(id)
    ); """

  cursor.execute(loan)
except Exception as e:
  print(f"Error: {e}")

conn.commit()

print("Tables created.")

# VERIFY YOUR TABLES
conn.execute("""SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;""").fetchall()


Tables created.


[('book',), ('loan',), ('member',)]

Load Book Data (Step 4)

In [5]:

with open(BOOK_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            #i edited the INSERT line " or ignore" beacsue i used an inconsistent table name
            #reloading it caused UNIQUE contraint errors
            'INSERT or ignore INTO book (callNo, title, author) VALUES (?, ?, ?);',
            (row['callNo'], row['title'], row['author'])
        )

conn.commit()
print('Book rows loaded:', conn.execute('SELECT COUNT(*) FROM Book;').fetchone()[0])

Book rows loaded: 11


Load Member Data (Step 5)

In [7]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
      #INSERT OR IGNORE
        conn.execute(
            'INSERT or ignore INTO member (id, firstname, lastName) VALUES (?, ?, ?);',
            (int(row['id']), row['firstname'], row['lastName'])
        )

conn.commit()
print('Member rows loaded:', conn.execute('SELECT COUNT(*) FROM Member;').fetchone()[0])

Member rows loaded: 4


Load Loan Data (Step 6)

In [8]:
with open(LOAN_PATH, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Convert empty dateReturned to None (→ NULL in SQLite)
        date_returned = row['dateReturned'] if row['dateReturned'].strip() else None
#INSERT OR IGNORE
        conn.execute(
            '''INSERT or ignore INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
               VALUES (?, ?, ?, ?, ?);''',

            (row['callNo'], int(row['id']),

             row['dateBorrowed'], date_returned, row['dateDue'])       )
        conn.commit()

print('Loan rows loaded:', conn.execute('SELECT COUNT(*) FROM Loan;').fetchone()[0])

Loan rows loaded: 4


QUERY 1 -- All Books Alphabetically Ordered

In [16]:
# QUERY 1: SOMEONE WNATED TO SEE THE BOOKS ALPHABETICALLY ORDERED BY AUTHOR
cursor= conn.execute( """
    select * from book
    order by author;
     """)
query1 = cursor.fetchall()
print("QUERY 1:")
print("All books alphabetically ordered by Author:")
for line in query1:
  print(line)

QUERY 1:
All books alphabetically ordered by Author:
('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


QUERY 2 -- BOOKS NOT YET TURNED. Fullnames of members with books not returned yet.

In [10]:
# OVERDUE BOOKS. THE LIBRARIAN ASKED ME TO MAKE A LIST OF ALL BOOKS NOT YET RETURNED
cursor=conn.execute ("""
SELECT book.title, members.firstname,members.lastName
FROM book
JOIN loan as loans on book.callNo = loans.callNo
JOIN member as members on members.id = loans.id
WHERE loans.dateReturned IS NULL
""")
query2 = cursor.fetchall()
print("All books yet to be returned:")
for line in query2:
  print(line)


All books yet to be returned:
("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


QUERY 3: Loan History For a Specific Book. What is all activity on callNo: R 141 E45 2006

In [13]:
cursor= conn.execute("""
SELECT member.firstname || ' ' || member.lastName as fullName,
loan.dateBorrowed, loan.dateDue, loan.dateReturned
FROM loan,member
JOIN member as members on loan.id = member.id

WHERE loan.callNo = 'R 141 E45 2006'
GROUP BY fullName
ORDER BY loan.dateBorrowed ASC
""")
query3 = cursor.fetchall()
print("Activity history of callNo: R 141 E45 2006")
for line in query3:
  print(line)

#QUERY 4: What are the full names of every member who does not have a loan on their account.

Activity history of callNo: R 141 E45 2006
('Betty Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


QUERY 4: Members Who Have Never Borrowed a Book. All members with no acitivty logged.

In [64]:

cursor = conn.execute("""
SELECT members.id, members.firstname, members.lastName
FROM member members
LEFT JOIN loan loans ON members.id = loans.id

WHERE loans.id IS NULL
""")

query4 = cursor.fetchall()
print("QUERY 4:\nMembers with no book activity logged:")
for line in query4:
  print(line)

QUERY 4:
Members with no book activity logged:
(4, 'John', 'Martin')


QUERY 5: Count of Loans Per Member. Fullnames of all members and their total count of rentals.

In [63]:
cursor = conn.execute("""
SELECT members.firstname || " " || members.lastName as fullName, COUNT(loans.dateBorrowed) as books
FROM member members
LEFT JOIN loan loans on members.id = loans.id

GROUP BY fullname
HAVING books >= 0
ORDER BY books desc
""")

query5 = cursor.fetchall()
print("QUERY 5:\nMembers book rental COUNT:")
for line in query5:
  print(line)

QUERY 5:
Members book rental COUNT:
('David Martin', 2)
('John Smith', 1)
('Betty Freeman', 1)
('John Martin', 0)


QUERY 6: FREE CHOICE ANALYTICAL -- After looking at the tables below, I decided to find the most popular book. I expect 'Medieval medicine and the plague' to be #1. Very small dataset.

In [130]:
# I will show the book title and the COUNT() amount of book rentals.
cursor = conn.execute("""
SELECT books.title, COUNT(loans.callNo) AS timesBorrowed
FROM book books
LEFT JOIN loan loans ON books.callNo = loans.callNo

GROUP BY books.title
ORDER BY timesBorrowed DESC
""")
print("QUERY 6:\nBOOK POPULARITY CONTEST")
for row in cursor:
  print(f"{row[0]:<10} | Borrowed {row[1]} times")
#I looked up formatting tips


QUERY 6:
BOOK POPULARITY CONTEST
Medieval medicine and the plague | Borrowed 2 times
Medieval women | Borrowed 1 times
Joe Celko's SQL puzzles & answers | Borrowed 1 times
Programming Android | Borrowed 0 times
Medieval miscellany | Borrowed 0 times
Medicine in medieval England. | Borrowed 0 times
Joe Celko's data & databases : concepts in practice | Borrowed 0 times
Joe Celko's Trees and hierarchies in SQL for smarties | Borrowed 0 times
Information modeling and relational databases | Borrowed 0 times
Data model patterns : conventions of thought | Borrowed 0 times
Atlas of medieval Europe | Borrowed 0 times


In [131]:
#SELECT members.firstname || " " || members.lastName as fullName, books.title, loans.dateBorrowed
print("LOAN COLUMNS:", [description[0] for description in conn.execute("SELECT * FROM loan LIMIT 1").description])
cursor = conn.execute("""
select * from loan""")
loaner = cursor.fetchall()
for line in loaner:
  print(line)
print()
print("MEMBER COLUMNS:", [description[0] for description in conn.execute("SELECT * FROM member LIMIT 1").description])
cursor=conn.execute("""
select * from member""")
memberer = cursor.fetchall()
for line in memberer:
  print(line)
print()

print("BOOK COLUMNS:", [description[0] for description in conn.execute("SELECT * FROM book limit 1").description])
cursor = conn.execute("""
select * from book""")
booker = cursor.fetchall()
for line in booker:
  print(line)

LOAN COLUMNS: ['callNo', 'id', 'dateBorrowed', 'dateReturned', 'dateDue']
('HQ 1143 P68 1975', 1, '5/1/2014 0:00', '5/3/2014 0:00', '5/15/2014 0:00')
('QA 76.73 S67C46 1997', 2, '5/4/2014 0:00', None, '5/18/2014 0:00')
('R 141 E45 2006', 2, '4/30/2014 0:00', None, '5/14/2014 0:00')
('R 141 E45 2006', 3, '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')

MEMBER COLUMNS: ['id', 'firstname', 'lastName']
(1, 'John', 'Smith')
(2, 'David', 'Martin')
(3, 'Betty', 'Freeman')
(4, 'John', 'Martin')

BOOK COLUMNS: ['callNo', 'title', 'author']
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')
('QA 76.9 D26H355 2008', 'Infor

SUMMARY:
I think the small dataset is a good size to practice on and play with different operations although it was toguh to be creative with it while wokring on QUERY 6. Ontop of that, a real library system will need more columns for better analytics. Some examples for the MEMBER table : AGE, ADDRESS, GENDER. Examples for BOOK table: GENRE, TOTAL PAGE#, AGE RATING. Overall, I think the dataset is a good size to work on and not get overwhlemed by rows/data.